In [1]:
print("Hello World")

Hello World


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Preparing RAG Evaluation Dataset

In [3]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Create dataset
qa_pairs = [
    {"question": q, "answer": a}
    for q, a in zip(inputs, outputs)
]
df = pd.DataFrame(qa_pairs)

# Write to CSV
csv_path = r"F:\AI Projects\End-to-End-LLMOPs-Project\data\goldens.csv"
df.to_csv(csv_path, index=False)

print(f"CSV saved successfully: {csv_path}")

CSV saved successfully: F:\AI Projects\End-to-End-LLMOPs-Project\data\goldens.csv


# Preparing RAG Evaluation Dataset in LangSmith

In [4]:
from langsmith import Client

client = Client()
dataset_name = "New2 Golden Dataset for RAG Evaluation"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['3a77cd08-eeef-4b36-a747-2a1bb7c656ba',
  '106b9a97-bb8d-48d9-8910-e1f22724b493',
  '3f6e882c-d056-4b67-8fd4-5860679a6f49'],
 'count': 3,
 'as_of': '2026-10-16T13:42:16.929936015Z'}

# Creating a LangSmith RAG Evaluation Function

In [5]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"F:\AI Projects\End-to-End-LLMOPs-Project")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG


class LocalFileAdapter:
    """Adapts a local file for document ingestion."""

    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name

    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = str(
        PROJECT_ROOT / "data" / "The 2025 AI Engineering Report.txt"
    ),
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """Answer AI Engineering Report questions using RAG."""

    try:
        question = inputs.get("question", "").strip()

        if not question:
            return {"answer": "No question provided."}

        file_path = Path(data_path)

        if not file_path.is_file():
            return {"answer": f"Data file not found: {file_path}"}

        ingestor = ChatIngestor(
            temp_base=str(PROJECT_ROOT / "data"),
            faiss_base=str(PROJECT_ROOT / "faiss_index"),
            use_session_dirs=True
        )

        ingestor.built_retriver(
            uploaded_files=[LocalFileAdapter(str(file_path))],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k,
            search_type="mmr",
            fetch_k=20,
            lambda_mult=0.5
        )

        session_id = ingestor.session_id
        index_path = PROJECT_ROOT / "faiss_index" / session_id

        rag = ConversationalRAG(session_id=session_id)

        rag.load_retriever_from_faiss(
            index_path=str(index_path),
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index"),
            search_type="mmr",
            fetch_k=20,
            lambda_mult=0.5
        )

        answer = rag.invoke(question, chat_history=[])

        return {"answer": answer}

    except Exception as e:
        return {"answer": f"Error: {str(e)}"} 

In [2]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-10-15T14:21:54.953803Z", "level": "info", "event": "Running in LOCAL mode"}
{"keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-10-15T14:21:54.956320Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20261015_195155_56940554", "temp_dir": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261015_195155_56940554", "faiss_dir": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\faiss_index\\session_20261015_195155_56940554", "sessionized": true, "timestamp": "2026-10-15T14:21:55.277230Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261015_195155_56940554\\the_2025_ai_engineering_report_71a13b.txt", "timestamp": "2026-10-15T14:21:55.464987Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-10-15T14:21:55.469644Z", "level": "info", "event": "Documents loaded"}
{"c

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: OpenAI models dominate the top rankings for customer‑facing applications.


In [5]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

{"timestamp": "2026-10-15T14:31:06.982766Z", "level": "info", "event": "Running in LOCAL mode"}
{"keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-10-15T14:31:06.986039Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20261015_200107_b226b29c", "temp_dir": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261015_200107_b226b29c", "faiss_dir": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\faiss_index\\session_20261015_200107_b226b29c", "sessionized": true, "timestamp": "2026-10-15T14:31:07.072518Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261015_200107_b226b29c\\the_2025_ai_engineering_report_29d5df.txt", "timestamp": "2026-10-15T14:31:07.079599Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-10-15T14:31:07.084180Z", "level": "info", "event": "Documents loaded"}
{"c

Testing all questions from the dataset:



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: OpenAI models dominate the top rankings for customer‑facing applications.

--------------------------------------------------------------------------------



{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261015_200123_e32bb8cd\\the_2025_ai_engineering_report_ed6dfd.txt", "timestamp": "2026-10-15T14:31:23.918432Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-10-15T14:31:23.934337Z", "level": "info", "event": "Documents loaded"}
{"chunks": 5, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-10-15T14:31:23.937208Z", "level": "info", "event": "Documents split"}
{"provider": "huggingface", "model": "sentence-transformers/all-MiniLM-L6-v2", "timestamp": "2026-10-15T14:31:23.939986Z", "level": "info", "event": "Loading embeddings"}
No device provided, using cpu
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48

Q2: What percentage of respondents are using RAG in some form?
A2: 70%.

--------------------------------------------------------------------------------



HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-tra

Q3: How often are most respondents updating their models?
A3: Most respondents update their models at least monthly, with about 17 % doing so weekly.

--------------------------------------------------------------------------------



# Evaluating RAG Performance in LangSmith


In [ ]:
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Evaluators
qa_evaluator = [LangChainStringEvaluator("cot_qa")]
dataset_name = "AgenticAIReportGoldens"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

# Custom Correctness Evaluator

### Creating an LLM-as-a-Judge evaluator to assess semantic and factual alignmen

In [ ]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }

# Run Evaluation with Custom Correctness Evaluator

In [ ]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "AgenticAIReportGoldens"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")

# Optional: Combine Multiple Evaluators

### You can use multiple evaluators together to get different perspectives on your RAG system's performance.

In [ ]:
# Example: Combine custom correctness evaluator with LangChain's built-in evaluators
from langsmith.evaluation import evaluate, LangChainStringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    LangChainStringEvaluator("cot_qa"),  # Chain-of-thought QA evaluator
]

# Run evaluation with multiple evaluators
# Uncomment to run:
# experiment_results_combined = evaluate(
#     answer_ai_report_question,
#     data=dataset_name,
#     evaluators=combined_evaluators,
#     experiment_prefix="agenticAIReport-multi-eval",
#     description="Evaluating RAG system with multiple evaluators",
#     metadata={
#         "variant": "RAG with FAISS",
#         "evaluators": "correctness + cot_qa",
#         "chunk_size": 1000,
#         "chunk_overlap": 200,
#         "k": 5,
#     },
# )